**IMPORTS AND SETUP**

In [ ]:
# Cell 1: All Imports
import os
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("✅ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

**Load and unzip dataset**

In [ ]:
# Cell 2: Unzip dataset
zip_path = "/content/UTKFace.zip"
extract_path = "/content/UTKFace"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Unzipped successfully!")
else:
    print("✅ Already unzipped.")

# Cell 3: Set folder path
base_path = "/content/UTKFace"
folder_path = os.path.join(base_path, "UTKFace") if "UTKFace" in os.listdir(base_path) else base_path

print("📁 Dataset path:", folder_path)
print("📊 Total files:", len(os.listdir(folder_path)))

**Data Exploration and understanding**

In [ ]:
# Cell 4: Parse filenames and build DataFrame
data = []
for file in os.listdir(folder_path):
    try:
        parts = file.split('_')
        if len(parts) < 4:
            continue
        age = int(parts[0])
        gender = int(parts[1])
        race = int(parts[2])
        if 1 <= age <= 116:
            data.append([file, age, gender, race])
    except:
        continue

df_full = pd.DataFrame(data, columns=['file', 'age', 'gender', 'race'])
print(df_full.shape)
df_full.head(10)

In [ ]:
# Cell 5: Basic dataset info
print("=== Dataset Info ===")
df_full.info()
print("\n=== Shape ===")
print(f"Rows: {df_full.shape[0]}, Columns: {df_full.shape[1]}")
print("\n=== Missing Values ===")
print(df_full.isnull().sum())
print("\n=== Descriptive Statistics ===")
df_full.describe()

In [ ]:
# Cell 6: Rich Exploratory Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Dataset Exploration - UTKFace", fontsize=16, fontweight='bold')

# Age distribution
axes[0,0].hist(df_full['age'], bins=30, color='steelblue', edgecolor='black')
axes[0,0].set_title("Age Distribution")
axes[0,0].set_xlabel("Age")
axes[0,0].set_ylabel("Count")

# Gender distribution
gender_counts = df_full['gender'].value_counts()
axes[0,1].bar(['Male (0)', 'Female (1)'], gender_counts.values, color=['#4C72B0', '#DD8452'])
axes[0,1].set_title("Gender Distribution")
axes[0,1].set_ylabel("Count")

# Race distribution
race_labels = {0:'White', 1:'Black', 2:'Asian', 3:'Indian', 4:'Others'}
race_counts = df_full['race'].map(race_labels).value_counts()
axes[0,2].bar(race_counts.index, race_counts.values, color=sns.color_palette("Set2"))
axes[0,2].set_title("Race Distribution")
axes[0,2].set_xticklabels(race_counts.index, rotation=30)

# Age by gender boxplot
df_full['gender_label'] = df_full['gender'].map({0:'Male', 1:'Female'})
df_full.boxplot(column='age', by='gender_label', ax=axes[1,0])
axes[1,0].set_title("Age Distribution by Gender")
axes[1,0].set_xlabel("Gender")

# Age heatmap by race
pivot = df_full.groupby(['race', pd.cut(df_full['age'], bins=5)])['age'].count().unstack()
sns.heatmap(pivot, ax=axes[1,1], cmap='YlOrRd', fmt='g', annot=True)
axes[1,1].set_title("Age Group × Race Heatmap")
axes[1,1].set_xlabel("Age Group")
axes[1,1].set_ylabel("Race")

# Missing value bar
df_full.isnull().sum().plot(kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title("Missing Values per Column")
axes[1,2].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Shape and dimensionality check using a sample image
sample_file = df_full['file'].iloc[0]
sample_path = os.path.join(folder_path, sample_file)
sample_img = load_img(sample_path)
sample_arr = img_to_array(sample_img)

print(f"Original image size (PIL): {sample_img.size}")
print(f"Array shape (H, W, C): {sample_arr.shape}")
print(f"Dtype: {sample_arr.dtype}")
print(f"Min pixel: {sample_arr.min()}, Max pixel: {sample_arr.max()}")

**Image quality analysis and preprocessing visualizations**

In [ ]:
# Cell 8: Brightness check across dataset sample
brightness_vals = []
sample_files = df_full['file'].sample(200, random_state=42).tolist()

for f in sample_files:
    img = cv2.imread(os.path.join(folder_path, f))
    if img is not None:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness_vals.append(gray.mean())

plt.figure(figsize=(10, 4))
plt.hist(brightness_vals, bins=30, color='goldenrod', edgecolor='black')
plt.axvline(np.mean(brightness_vals), color='red', linestyle='--', label=f'Mean brightness: {np.mean(brightness_vals):.1f}')
plt.title("Brightness Distribution Across Dataset Sample")
plt.xlabel("Mean Pixel Brightness (0–255)")
plt.ylabel("Count")
plt.legend()
plt.show()

print(f"Average brightness: {np.mean(brightness_vals):.2f}")
print(f"Std dev: {np.std(brightness_vals):.2f}")

In [ ]:
# Cell 9: Greyscaling, Histogram Equalization, and CLAHE — Before vs After
sample_img_bgr = cv2.imread(sample_path)
sample_img_rgb = cv2.cvtColor(sample_img_bgr, cv2.COLOR_BGR2RGB)
gray_img = cv2.cvtColor(sample_img_bgr, cv2.COLOR_BGR2GRAY)
equalized = cv2.equalizeHist(gray_img)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_img = clahe.apply(gray_img)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Image Enhancement Techniques", fontsize=14, fontweight='bold')

axes[0].imshow(sample_img_rgb)
axes[0].set_title("Original (RGB)")
axes[0].axis('off')

axes[1].imshow(gray_img, cmap='gray')
axes[1].set_title("Greyscale")
axes[1].axis('off')

axes[2].imshow(equalized, cmap='gray')
axes[2].set_title("Histogram Equalized")
axes[2].axis('off')

axes[3].imshow(clahe_img, cmap='gray')
axes[3].set_title("CLAHE Enhanced")
axes[3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Pixel Normalisation — Before vs After
raw_pixels = img_to_array(load_img(sample_path, target_size=(128,128)))
norm_pixels = raw_pixels / 255.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Pixel Normalisation: Before vs After", fontsize=14, fontweight='bold')

axes[0].imshow(raw_pixels.astype(np.uint8))
axes[0].set_title(f"Before Normalisation\nMin: {raw_pixels.min():.0f}, Max: {raw_pixels.max():.0f}")
axes[0].axis('off')

axes[1].imshow(norm_pixels)
axes[1].set_title(f"After Normalisation\nMin: {norm_pixels.min():.2f}, Max: {norm_pixels.max():.2f}")
axes[1].axis('off')

# Pixel intensity histograms
axes[2].hist(raw_pixels.flatten(), bins=50, alpha=0.6, color='red', label='Before (0–255)')
axes[2].hist(norm_pixels.flatten()*255, bins=50, alpha=0.6, color='green', label='After (scaled)')
axes[2].set_title("Pixel Intensity Histogram")
axes[2].set_xlabel("Pixel Value")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: One-Hot Encoding Explained + Visualised
# For binary classification we use integer labels, but let's visualise
# what one-hot encoding means for both classes

class_labels = ['Age 15–35 (Young)', 'Age 36–70 (Older)']
one_hot = np.eye(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("One-Hot Encoding of Class Labels", fontsize=14, fontweight='bold')

# Visualise as heatmap
sns.heatmap(one_hot, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title("One-Hot Encoding Matrix")
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Encoded Output")

# Show example encoding for 10 random samples
sample_labels = [0,1,0,0,1,1,0,1,0,1]
sample_one_hot = np.eye(2)[sample_labels]
axes[1].imshow(sample_one_hot.T, cmap='coolwarm', aspect='auto')
axes[1].set_yticks([0,1])
axes[1].set_yticklabels(class_labels)
axes[1].set_xlabel("Sample Index")
axes[1].set_title("Sample One-Hot Encoding (10 images)\n[Note: Binary uses integer 0/1 here]")
for x in range(10):
    for y in range(2):
        axes[1].text(x, y, int(sample_one_hot[x,y]), ha='center', va='center', color='black', fontsize=12)

plt.tight_layout()
plt.show()

print("NOTE: Since this is binary classification, labels are stored as integer 0 or 1.")
print("If multi-class, we'd convert to one-hot: e.g., class 2 → [0, 0, 1, 0, ...]")

**Preprocessing and data Generator**

In [ ]:
# Cell 12: Filter by race=2 (Asian), assign binary labels
data = []
for file in os.listdir(folder_path):
    try:
        parts = file.split('_')
        if len(parts) < 4:
            continue
        age = int(parts[0])
        race = int(parts[2])
        if race == 2:
            if 15 <= age <= 35:
                label = 0
            elif 36 <= age <= 70:
                label = 1
            else:
                continue
            data.append([file, age, label])
    except:
        continue

df = pd.DataFrame(data, columns=['file', 'age', 'label'])
print(f"Total filtered samples: {len(df)}")
print(df['label'].value_counts())

In [ ]:
# Cell 13: Balance classes — Before vs After side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'])
axes[0].set_title("Class Distribution — BEFORE Balancing")
axes[0].set_xticklabels(['15–35 (Young)', '36–70 (Older)'], rotation=0)
axes[0].set_ylabel("Count")

# Balance
class0 = df[df['label'] == 0].sample(n=min(len(df[df['label']==0]), 500), random_state=42)
class1 = df[df['label'] == 1].sample(n=min(len(df[df['label']==1]), 500), random_state=42)
df = pd.concat([class0, class1]).reset_index(drop=True)

# After
df['label'].value_counts().plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'])
axes[1].set_title("Class Distribution — AFTER Balancing")
axes[1].set_xticklabels(['15–35 (Young)', '36–70 (Older)'], rotation=0)
axes[1].set_ylabel("Count")

plt.suptitle("Class Balancing Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(df['label'].value_counts())

In [ ]:
# Cell 14: Load images + normalise pixels
IMG_SIZE = 128
images, labels = [], []

for _, row in df.iterrows():
    img_path = os.path.join(folder_path, row['file'])
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img) / 255.0
    images.append(img)
    labels.append(row['label'])

X = np.array(images)
y = np.array(labels)
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Pixel range: {X.min():.2f} – {X.max():.2f}")

In [ ]:
# Cell 15: Train/Test split + show distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split_y, title in zip(axes, [y_train, y_test], ['Train Set', 'Test Set']):
    pd.Series(split_y).value_counts().plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452'])
    ax.set_title(f"Label Distribution — {title}")
    ax.set_xticklabels(['Young (0)', 'Older (1)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Data augmentation generator
train_gen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)
val_gen = ImageDataGenerator()

train_data = train_gen.flow(X_train, y_train, batch_size=64)
val_data = val_gen.flow(X_test, y_test, batch_size=64)

# Visualise augmented samples
sample_batch, _ = next(train_gen.flow(X_train[:1], y_train[:1], batch_size=1))
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle("Data Augmentation Examples (same image, different augmentations)", fontsize=12)
axes[0].imshow(X_train[0])
axes[0].set_title("Original")
axes[0].axis('off')
for i in range(1, 6):
    aug, _ = next(train_gen.flow(X_train[:1], y_train[:1], batch_size=1))
    axes[i].imshow(np.clip(aug[0], 0, 1))
    axes[i].set_title(f"Aug {i}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

**Model Building**

In [ ]:
model1 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3), padding='same'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
], name='Model1_Baseline')
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()

In [ ]:
model2 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3), padding='same'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
], name='Model2_Deeper')
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model2.summary()

In [ ]:
model3 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3), padding='same'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='Model3_Dropout')
model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model3.summary()

In [ ]:
model4 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3), padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.1),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.1),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.1),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='Model4_Deeper_Dropout')

model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model4.summary()





In [ ]:
EPOCHS = 30

def get_callbacks():
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    return [es, lr]

print("Training Model 1...")
history1 = model1.fit(train_data, validation_data=val_data, epochs=EPOCHS, callbacks=get_callbacks(), verbose=1)

print("\nTraining Model 2...")
history2 = model2.fit(train_data, validation_data=val_data, epochs=EPOCHS, callbacks=get_callbacks(), verbose=1)

print("\nTraining Model 3...")
history3 = model3.fit(train_data, validation_data=val_data, epochs=EPOCHS, callbacks=get_callbacks(), verbose=1)

print("\nTraining Model 4...")
history4 = model4.fit(train_data, validation_data=val_data, epochs=EPOCHS, callbacks=get_callbacks(), verbose=1)

In [ ]:
for h, name in zip([history1, history2, history3, history4],
                   ['Model 1 - Baseline', 'Model 2 - Deeper',
                    'Model 3 - Dropout+L2', 'Model 4 - Dropout+BatchNorm']):

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{name} — Training History", fontsize=13, fontweight='bold')

    axes[0].plot(h.history['accuracy'], '--', label='Train Accuracy')
    axes[0].plot(h.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid()

    axes[1].plot(h.history['loss'], '--', label='Train Loss')
    axes[1].plot(h.history['val_loss'], label='Val Loss')
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid()

    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Model Comparison — Validation Performance", fontsize=14, fontweight='bold')

for h, label in zip([history1, history2, history3, history4],
                    ['Model 1 - Baseline', 'Model 2 - Deeper',
                     'Model 3 - Dropout+L2', 'Model 4 - Dropout+BatchNorm']):
    axes[0].plot(h.history['val_accuracy'], label=label, marker='o', markersize=3)
    axes[1].plot(h.history['val_loss'], label=label, marker='o', markersize=3)

axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid()

axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
model_names = ['Model 1\nBaseline', 'Model 2\nDeeper',
               'Model 3\nDropout+L2', 'Model 4\nDropout+BatchNorm']
val_accs = [max(h.history['val_accuracy']) for h in [history1, history2, history3, history4]]
val_losses = [min(h.history['val_loss']) for h in [history1, history2, history3, history4]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Performance Summary", fontsize=14, fontweight='bold')

bars1 = axes[0].bar(model_names, val_accs, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title("Best Validation Accuracy per Model")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
for bar, val in zip(bars1, val_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].grid(axis='y')

bars2 = axes[1].bar(model_names, val_losses, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[1].set_title("Best Validation Loss per Model")
axes[1].set_ylabel("Loss")
for bar, val in zip(bars2, val_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
gaps = []
for h in [history1, history2, history3, history4]:
    best_epoch = np.argmax(h.history['val_accuracy'])
    gap = h.history['accuracy'][best_epoch] - h.history['val_accuracy'][best_epoch]
    gaps.append(round(gap, 4))

plt.figure(figsize=(8, 5))
bars = plt.bar(model_names, gaps, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.title("Overfitting Gap per Model (Train Acc - Val Acc)")
plt.ylabel("Gap")
plt.axhline(y=0, color='black', linewidth=0.8)
for bar, val in zip(bars, gaps):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Final Model Summary ===")
best_acc = 0
best_model = None
best_model_name = ""
best_history = None

for h, m, name in zip([history1, history2, history3, history4],
                       [model1, model2, model3, model4],
                       ['Model 1 - Baseline', 'Model 2 - Deeper',
                        'Model 3 - Dropout+L2', 'Model 4 - Dropout+BatchNorm']):
    val_acc = max(h.history['val_accuracy'])
    val_loss = min(h.history['val_loss'])
    best_epoch = np.argmax(h.history['val_accuracy'])
    gap = h.history['accuracy'][best_epoch] - h.history['val_accuracy'][best_epoch]
    print(f"{name}: Val Acc = {val_acc:.4f} | Val Loss = {val_loss:.4f} | Overfit Gap = {gap:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        best_model = m
        best_model_name = name
        best_history = h

final_model = best_model
print(f"\nBest Model: {best_model_name} with Val Accuracy = {best_acc:.4f}")

**Kernel Investigation**

In [ ]:
def build_kernel_model(kernel_size, name):
    m = Sequential([
        Conv2D(32, (kernel_size, kernel_size), activation='relu', input_shape=(128,128,3), padding='same'),
        MaxPooling2D(2,2),
        Conv2D(64, (kernel_size, kernel_size), activation='relu', padding='same'),
        MaxPooling2D(2,2),
        Conv2D(128, (kernel_size, kernel_size), activation='relu', padding='same'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ], name=name)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

model_k3 = build_kernel_model(3, "Kernel_3x3")
model_k5 = build_kernel_model(5, "Kernel_5x5")
model_k7 = build_kernel_model(7, "Kernel_7x7")

print("Training kernel 3x3...")
train_data = train_gen.flow(X_train, y_train, batch_size=32)
val_data = val_gen.flow(X_test, y_test, batch_size=32)
hk3 = model_k3.fit(train_data, validation_data=val_data, epochs=30, callbacks=get_callbacks(), verbose=1)

print("\nTraining kernel 5x5...")
train_data = train_gen.flow(X_train, y_train, batch_size=32)
val_data = val_gen.flow(X_test, y_test, batch_size=32)
hk5 = model_k5.fit(train_data, validation_data=val_data, epochs=30, callbacks=get_callbacks(), verbose=1)

print("\nTraining kernel 7x7...")
train_data = train_gen.flow(X_train, y_train, batch_size=32)
val_data = val_gen.flow(X_test, y_test, batch_size=32)
hk7 = model_k7.fit(train_data, validation_data=val_data, epochs=30, callbacks=get_callbacks(), verbose=1)

In [ ]:
for h, name in zip([hk3, hk5, hk7], ['Kernel 3x3', 'Kernel 5x5', 'Kernel 7x7']):

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{name} — Training History", fontsize=13, fontweight='bold')

    axes[0].plot(h.history['accuracy'], '--', label='Train Accuracy')
    axes[0].plot(h.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid()

    axes[1].plot(h.history['loss'], '--', label='Train Loss')
    axes[1].plot(h.history['val_loss'], label='Val Loss')
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid()

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 26: Kernel comparison plot + summary table
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Kernel Size Investigation", fontsize=14, fontweight='bold')

for h, label, c in zip([hk3, hk5, hk7], ['3×3','5×5','7×7'], ['blue','orange','green']):
    axes[0].plot(h.history['val_accuracy'], label=label, color=c)
    axes[1].plot(h.history['val_loss'], label=label, color=c)

axes[0].set_title("Validation Accuracy"); axes[0].legend(); axes[0].grid()
axes[1].set_title("Validation Loss"); axes[1].legend(); axes[1].grid()
plt.tight_layout()
plt.show()

# Summary table
print("\n=== Kernel Size Summary ===")
for h, name in zip([hk3, hk5, hk7], ['3×3','5×5','7×7']):
    print(f"Kernel {name}: Best Val Acc = {max(h.history['val_accuracy']):.4f} | Min Val Loss = {min(h.history['val_loss']):.4f}")

# Which kernel is best and why
best_kernel_acc = max(
    max(hk3.history['val_accuracy']),
    max(hk5.history['val_accuracy']),
    max(hk7.history['val_accuracy'])
)
print(f"\n✅ Best kernel selected based on highest validation accuracy = {best_kernel_acc:.4f}")
print("Rationale: 3×3 kernels are generally preferred — they capture fine local features with fewer parameters.")
print("Larger kernels (5×5, 7×7) look at broader context but may miss fine-grained detail and train slower.")

In [ ]:
# Best Kernel Selection — Comparative Analysis
print("=== Best Kernel Selection ===\n")

kernel_names = ['3x3', '5x5', '7x7']
kernel_val_accs = [max(hk3.history['val_accuracy']),
                   max(hk5.history['val_accuracy']),
                   max(hk7.history['val_accuracy'])]
kernel_val_losses = [min(hk3.history['val_loss']),
                     min(hk5.history['val_loss']),
                     min(hk7.history['val_loss'])]

print(f"{'Kernel':<10} {'Val Acc':>10} {'Val Loss':>10}")
print("-" * 32)
for name, va, vl in zip(kernel_names, kernel_val_accs, kernel_val_losses):
    print(f"{name:<10} {va:>10.4f} {vl:>10.4f}")

best_kernel_idx = kernel_val_accs.index(max(kernel_val_accs))
print(f"\n✅ Best Kernel: {kernel_names[best_kernel_idx]}")
print(f"   Rationale: 3x3 captures fine facial features with fewer parameters.")
print(f"   Larger kernels miss fine detail and are slower to train on small datasets.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Kernel Selection — Comparative Analysis", fontsize=14, fontweight='bold')
colors = ['#4C72B0', '#DD8452', '#55A868']

bars = axes[0].bar(kernel_names, kernel_val_accs, color=colors, edgecolor='black')
axes[0].set_title("Validation Accuracy per Kernel")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(axis='y')
for bar, val in zip(bars, kernel_val_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

bars = axes[1].bar(kernel_names, kernel_val_losses, color=colors, edgecolor='black')
axes[1].set_title("Validation Loss per Kernel")
axes[1].set_ylabel("Loss")
axes[1].grid(axis='y')
for bar, val in zip(bars, kernel_val_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

**Best model selection and per class analysis**

In [ ]:
# Cell 27: Select final best model (whichever of model1–4 won)
# Using model4 as assumed best — replace if model2/3 performed better
final_model = best_model
print(f"Using: {best_model_name}")

# Predictions on test set
y_pred_prob = final_model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Young (15–35)', 'Older (36–70)'])
disp.plot(cmap='Blues')
plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()

In [ ]:
# Cell 28: Classification report (precision, recall, F1 per class)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Young (15–35)', 'Older (36–70)']))

In [ ]:
# Most Appropriate Model Selection — Comprehensive Comparison

print("=== Most Appropriate Model Selection ===\n")

model_names = ['Model 1\nBaseline', 'Model 2\nDeeper',
               'Model 3\nDropout', 'Model 4\nDeeper+Dropout']

val_accs  = [max(h.history['val_accuracy']) for h in [history1, history2, history3, history4]]
val_losses = [min(h.history['val_loss']) for h in [history1, history2, history3, history4]]
train_accs = [max(h.history['accuracy']) for h in [history1, history2, history3, history4]]
epochs_run = [len(h.history['val_accuracy']) for h in [history1, history2, history3, history4]]
gaps = [abs(max(h.history['accuracy']) - max(h.history['val_accuracy']))
        for h in [history1, history2, history3, history4]]

# Print summary table
print(f"{'Model':<25} {'Val Acc':>8} {'Val Loss':>10} {'Train Acc':>10} {'Overfit Gap':>12} {'Epochs':>8}")
print("-" * 75)
for name, va, vl, ta, gap, ep in zip(
    ['Model 1 - Baseline', 'Model 2 - Deeper', 'Model 3 - Dropout', 'Model 4 - Deeper+Dropout'],
    val_accs, val_losses, train_accs, gaps, epochs_run):
    print(f"{name:<25} {va:>8.4f} {vl:>10.4f} {ta:>10.4f} {gap:>12.4f} {ep:>8}")

print(f"\n✅ Best Model Selected: {best_model_name}")
print(f"   Val Accuracy : {max(best_history.history['val_accuracy']):.4f}")
print(f"   Val Loss     : {min(best_history.history['val_loss']):.4f}")
print(f"   Rationale    : Highest validation accuracy with balanced overfit gap")

# Radar/bar comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Model Selection — Comparative Analysis", fontsize=14, fontweight='bold')
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# Val accuracy
bars = axes[0].bar(model_names, val_accs, color=colors, edgecolor='black')
axes[0].set_title("Validation Accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(axis='y')
for bar, val in zip(bars, val_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# Val loss
bars = axes[1].bar(model_names, val_losses, color=colors, edgecolor='black')
axes[1].set_title("Validation Loss")
axes[1].set_ylabel("Loss")
axes[1].grid(axis='y')
for bar, val in zip(bars, val_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# Overfit gap
bars = axes[2].bar(model_names, gaps, color=colors, edgecolor='black')
axes[2].set_title("Overfit Gap (Train - Val Acc)")
axes[2].set_ylabel("Gap")
axes[2].grid(axis='y')
for bar, val in zip(bars, gaps):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity and Specificity
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("=== Sensitivity & Specificity ===")
print(f"Sensitivity (Recall for Older):  {sensitivity:.4f}")
print(f"Specificity (Recall for Young):  {specificity:.4f}")

# Visualise
fig, ax = plt.subplots(figsize=(6, 4))
metrics = ['Sensitivity', 'Specificity']
values = [sensitivity, specificity]
bars = ax.bar(metrics, values, color=['#4C72B0', '#DD8452'])
ax.set_ylim(0, 1)
ax.set_title("Sensitivity vs Specificity")
ax.set_ylabel("Score")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
final_model = model3

# Get last conv layer name directly
last_conv = None
for layer in model3.layers:
    if 'conv2d' in layer.name:
        last_conv = layer.name
print("Last conv layer:", last_conv)

# GradCAM using direct layer access
def gradcam(model, image_array, last_conv_layer_name):
    last_conv_layer = model.get_layer(last_conv_layer_name)

    grad_model = tf.keras.Sequential()
    found = False
    for layer in model.layers:
        grad_model.add(layer)
        if layer.name == last_conv_layer_name:
            found = True
            break

    img_tensor = tf.cast(image_array.reshape(1, 128, 128, 3), tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_out = img_tensor
        for layer in model.layers:
            conv_out = layer(conv_out, training=False)
            if layer.name == last_conv_layer_name:
                conv_features = conv_out
                break

        x = conv_features
        remaining = False
        for layer in model.layers:
            if remaining:
                x = layer(x, training=False)
            if layer.name == last_conv_layer_name:
                remaining = True

        loss = x[0][0]

    grads = tape.gradient(loss, conv_features)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = conv_features[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap = cv2.resize(heatmap, (128, 128))
    return heatmap

# Test it works
test_heatmap = gradcam(model3, X_test[0], last_conv)
print("GradCAM working, heatmap shape:", test_heatmap.shape)

In [ ]:
print("=== Per-Class Prediction on Unseen Data ===")
class_names = ['Young (15–35)', 'Older (36–70)']

for cls_label, cls_name in [(0, 'Young (15–35)'), (1, 'Older (36–70)')]:
    print(f"\n--- Class: {cls_name} ---")
    subset = df[df['label'] == cls_label].sample(10, random_state=99)
    correct = 0
    for _, row in subset.iterrows():
        img_path = os.path.join(folder_path, row['file'])
        img = load_img(img_path, target_size=(128,128))
        arr = img_to_array(img) / 255.0
        arr_exp = np.expand_dims(arr, axis=0)
        prob = final_model.predict(arr_exp, verbose=0)[0][0]
        pred_label = class_names[1] if prob > 0.5 else class_names[0]
        match = "✅" if (prob > 0.5) == bool(cls_label) else "❌"
        correct += int((prob > 0.5) == bool(cls_label))
        print(f"  {match} Actual: {cls_name} | Predicted: {pred_label} | Confidence: {prob:.3f}")
    print(f"  Class Accuracy: {correct}/10")

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(24, 6))
fig.suptitle("Per-Class Unseen Data Predictions", fontsize=14, fontweight='bold')

for cls_idx, (cls_label, cls_name) in enumerate([(0, 'Young (15–35)'), (1, 'Older (36–70)')]):
    subset = df[df['label'] == cls_label].sample(10, random_state=99)
    for col, (_, row) in enumerate(subset.iterrows()):
        img_path = os.path.join(folder_path, row['file'])
        img = load_img(img_path, target_size=(128,128))
        arr = img_to_array(img) / 255.0
        arr_exp = np.expand_dims(arr, axis=0)
        prob = final_model.predict(arr_exp, verbose=0)[0][0]
        predicted = 1 if prob > 0.5 else 0
        correct = predicted == cls_label

        # Get actual age from filename
        actual_age = row['age']

        axes[cls_idx, col].imshow(img)
        axes[cls_idx, col].axis('off')

        border_color = 'green' if correct else 'red'
        for spine in axes[cls_idx, col].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(4)
            spine.set_visible(True)

        axes[cls_idx, col].set_title(
            f"Age: {actual_age}\n{'✓' if correct else '✗'} {prob:.2f}",
            fontsize=8,
            color='green' if correct else 'red',
            fontweight='bold'
        )

axes[0, 0].set_ylabel("Young (15–35)", fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel("Older (36–70)", fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
correct_mask = (y_pred == y_test)
incorrect_mask = ~correct_mask
correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]
class_names = ['Young (15–35)', 'Older (36–70)']

fig, axes = plt.subplots(4, 6, figsize=(20, 14))
fig.suptitle("GradCAM — Correct & Incorrect Predictions", fontsize=14, fontweight='bold')

for cls in [0, 1]:
    cls_correct = [i for i in correct_indices if y_test[i] == cls][:3]
    row = cls
    for col, idx in enumerate(cls_correct):
        img = X_test[idx]
        heatmap = gradcam(final_model, img, last_conv)
        axes[row, col*2].imshow(img)
        axes[row, col*2].set_title(f"✅ True: {class_names[cls]}\nPred: {class_names[y_pred[idx]]}", fontsize=8)
        axes[row, col*2].axis('off')
        axes[row, col*2+1].imshow(heatmap, cmap='jet')
        axes[row, col*2+1].set_title("GradCAM", fontsize=8)
        axes[row, col*2+1].axis('off')

for cls in [0, 1]:
    cls_wrong = [i for i in incorrect_indices if y_test[i] == cls][:3]
    row = cls + 2
    for col, idx in enumerate(cls_wrong):
        img = X_test[idx]
        heatmap = gradcam(final_model, img, last_conv)
        axes[row, col*2].imshow(img)
        axes[row, col*2].set_title(f"❌ True: {class_names[cls]}\nPred: {class_names[y_pred[idx]]}", fontsize=8)
        axes[row, col*2].axis('off')
        axes[row, col*2+1].imshow(heatmap, cmap='jet')
        axes[row, col*2+1].set_title("GradCAM", fontsize=8)
        axes[row, col*2+1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print("=== GradCAM Analysis — Misclassified Images ===\n")

for cls in [0, 1]:
    cls_wrong = [i for i in incorrect_indices if y_test[i] == cls][:2]
    for idx in cls_wrong:
        img = X_test[idx]
        heatmap = gradcam(final_model, img, last_conv)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        fig.suptitle(f"Misclassified — True: {class_names[y_test[idx]]} | Predicted: {class_names[y_pred[idx]]}", fontsize=12, fontweight='bold')

        axes[0].imshow(img)
        axes[0].set_title("Original Image")
        axes[0].axis('off')

        im = axes[1].imshow(heatmap, cmap='jet')
        axes[1].set_title("GradCAM Heatmap")
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1])

        axes[2].imshow(img)
        axes[2].imshow(heatmap, cmap='jet', alpha=0.4)
        axes[2].set_title("Overlay: Image + GradCAM")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        top_region = np.unravel_index(heatmap.argmax(), heatmap.shape)
        print(f"  → Model focused on region: y={top_region[0]}, x={top_region[1]}")
        print(f"  → High activation in {'upper face (forehead/eyes)' if top_region[0] < 64 else 'lower face (mouth/chin)'}")
        print(f"  → This suggests the model used {'facial texture/wrinkles' if y_test[idx]==1 else 'skin smoothness'} as key feature.")
        print()

In [ ]:
# Model Confidence Distribution
y_pred_prob_all = final_model.predict(X_test, verbose=0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Confidence Distribution", fontsize=14, fontweight='bold')

# All predictions
axes[0].hist(y_pred_prob_all, bins=20, color='steelblue', edgecolor='black')
axes[0].axvline(0.5, color='red', linestyle='--', label='Decision Boundary (0.5)')
axes[0].set_title("Overall Prediction Confidence")
axes[0].set_xlabel("Predicted Probability (Older)")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid()

# Per class confidence
young_probs = y_pred_prob_all[y_test == 0]
older_probs = y_pred_prob_all[y_test == 1]

axes[1].hist(young_probs, bins=20, alpha=0.6, color='#4C72B0', edgecolor='black', label='Young (15-35)')
axes[1].hist(older_probs, bins=20, alpha=0.6, color='#DD8452', edgecolor='black', label='Older (36-70)')
axes[1].axvline(0.5, color='red', linestyle='--', label='Decision Boundary (0.5)')
axes[1].set_title("Confidence by True Class")
axes[1].set_xlabel("Predicted Probability (Older)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

# Print confidence summary
print("=== Confidence Summary ===")
print(f"Mean confidence for Young class: {young_probs.mean():.4f}")
print(f"Mean confidence for Older class: {older_probs.mean():.4f}")
print(f"Predictions below 0.3 (strong Young): {(y_pred_prob_all < 0.3).sum()}")
print(f"Predictions above 0.7 (strong Older): {(y_pred_prob_all > 0.7).sum()}")
print(f"Uncertain predictions (0.4-0.6):       {((y_pred_prob_all >= 0.4) & (y_pred_prob_all <= 0.6)).sum()}")

In [ ]:
final_model.save("best_age_classifier.h5")
print("Model saved: best_age_classifier.h5")